# Phase 2 smoke verification (30s)

**Goal:** confirm this chain on a Colab NVIDIA GPU:

`GPU → TinyStories → tokenizer → train → val_bpb → checkpoint → result JSON → checkpoint reload`

1. Runtime → Change runtime type → **T4/L4/A100 GPU**
2. Run all cells in order
3. Final cell must print `SMOKE CHAIN: PASS`

Default training budget for this notebook: **30 seconds** (`AUTORESEARCH_TIME_BUDGET=30`).  
Production experiments still use `TIME_BUDGET=300` in `prepare.py`.

In [ ]:
# STEP 1 — CUDA
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing"
print("PASS STEP 1 CUDA:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# STEP 2 — clone / update repo
import os
from pathlib import Path

REPO = "autoresearch-platform"
URL = "https://github.com/justSamarthings/autoresearch-platform.git"

if Path(REPO).exists():
    %cd {REPO}
    !git fetch origin
    !git checkout main
    !git pull --ff-only origin main
else:
    !git clone {URL}
    %cd {REPO}

print("PASS STEP 2 REPO:", Path.cwd(), "|", end=" ")
!git rev-parse --short HEAD

In [ ]:
# STEP 3 — training deps (Torch comes from Colab)
%cd training
!pip install -q -r requirements.txt
# Optional FA3; train.py falls back to SDPA if this fails
!pip install -q "kernels>=0.11.7" || true
print("PASS STEP 3 DEPS")

In [ ]:
# STEP 4 — TinyStories prepare + tokenizer (~673MB download first time)
!python prepare.py

from pathlib import Path
import os

cache = Path(os.environ.get("AUTORESEARCH_CACHE", Path.home() / ".cache" / "autoresearch-platform"))
train_pq = cache / "data" / "train.parquet"
val_pq = cache / "data" / "val.parquet"
tok = cache / "tokenizer" / "tokenizer.pkl"
tb = cache / "tokenizer" / "token_bytes.pt"
assert train_pq.exists() and val_pq.exists(), f"Missing TinyStories splits under {cache}/data"
assert tok.exists() and tb.exists(), f"Missing tokenizer under {cache}/tokenizer"
print("PASS STEP 4 TINYSTORIES+TOKENIZER:", cache)

In [ ]:
# STEP 5 — 30s smoke train (env must be on the shell line for !python)
!AUTORESEARCH_TIME_BUDGET=30 AUTORESEARCH_NO_COMPILE=1 python train.py
print("PASS STEP 5 TRAIN finished (see val_bpb summary above)")

In [ ]:
# STEP 6 — val_bpb + checkpoint + JSON + reload
from pathlib import Path
import json
import subprocess
import sys

ckpt_dir = Path("artifacts/checkpoints")
res_dir = Path("artifacts/results")
ckpts = sorted(ckpt_dir.glob("*.pt"))
results = sorted(res_dir.glob("*.json"))
assert ckpts, "No checkpoint under artifacts/checkpoints"
assert results, "No result JSON under artifacts/results"

ckpt = ckpts[-1]
result_path = results[-1]
result = json.loads(result_path.read_text())

assert result.get("status") == "ok", result
assert result.get("val_bpb") is not None, "val_bpb missing from result JSON"
assert isinstance(result["val_bpb"], (int, float)), result["val_bpb"]
assert result.get("checkpoint_path"), "checkpoint_path missing"
assert Path(result["checkpoint_path"]).exists() or ckpt.exists()

print("PASS STEP 6a val_bpb:", result["val_bpb"])
print("PASS STEP 6b checkpoint:", ckpt)
print("PASS STEP 6c result JSON:", result_path)

proc = subprocess.run(
    [sys.executable, "../scripts/verify_checkpoint.py", str(ckpt)],
    check=False,
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise SystemExit(f"checkpoint reload failed with code {proc.returncode}")
print("PASS STEP 6d checkpoint reload")
print("\nSMOKE CHAIN: PASS")

### Optional later: full 5-minute experiment

Only after the smoke chain passes:

```python
!python train.py
```